# 🛰 Synthetic Satellite Environment

This notebook demonstrates the `SyntheticEnvironment` — a step-controlled,
Jupyter-friendly live satellite simulation built on top of the project's
real SGP4 propagation stack.

**What you can do:**
- Load real TLE files (Starlink, OneWeb, Kuiper) **or** use the built-in synthetic fleets
- Step the simulation clock forward by a configurable amount
- Query satellite positions, velocities, and active conjunctions at any step
- Export data as NumPy arrays or a pandas DataFrame

In [ ]:
import sys, os
# If running from the notebooks/ subdirectory, point Python at the src package
sys.path.insert(0, os.path.join('..', 'src'))

from satcollision.environment import SyntheticEnvironment

## 1. Build the environment

### Option A — Real TLE files (recommended)

Uses the TLE files you already downloaded into `data/`.  
Setting `inject_encounter=True` also adds one engineered close-approach pair
so there is always at least one guaranteed conjunction for demonstration.

In [ ]:
DATA = os.path.join('..', 'data')   # path to the data/ directory relative to this notebook

tle_files = {}
for fname, op in [('starlink.tle', 'Starlink'), ('kuiper.tle', 'Kuiper'), ('oneweb.tle', 'OneWeb')]:
    p = os.path.join(DATA, fname)
    if os.path.exists(p):
        tle_files[op] = p
        print(f'Found {fname} → operator "{op}"')
    else:
        print(f'Not found: {fname} (will skip)')

if tle_files:
    env = SyntheticEnvironment.from_tle_files(
        tle_files       = tle_files,
        inject_encounter= True,    # always guarantee a conjunction exists for demo
        step_size_s     = 300.0,   # 5-minute steps
        screening_window_s = 3600.0,
        screening_step_s   = 30.0,
    )
else:
    print('No TLE files found — falling back to synthetic fleets.')
    env = SyntheticEnvironment.from_synthetic_fleets(inject_encounter=True, step_size_s=300.0)

env   # auto-renders via _repr_html_

### Option B — Pure synthetic fleets (no TLE files needed)

Uncomment the cell below to use Alpha/Beta/Gamma + debris fleets instead.

In [ ]:
# env = SyntheticEnvironment.from_synthetic_fleets(
#     inject_encounter = True,
#     step_size_s      = 60.0,
# )
# env

## 2. Initial snapshot (time does NOT advance)

In [ ]:
snap = env.snapshot()
print(snap.summary())

## 3. Step the simulation forward

In [ ]:
snap = env.step()   # advance by one step (300 s)
env.display()       # rich colour console output (works in both Jupyter and terminal)

In [ ]:
snap = env.step(n=5)   # advance 5 more steps (25 min)
env                    # auto-renders HTML table

## 4. Extract raw data

### 4a. NumPy position & velocity arrays

In [ ]:
positions, norad_ids = env.positions_array()   # (n_sats, 3) km ECI
velocities, _        = env.velocities_array()  # (n_sats, 3) km/s ECI

print(f'positions  shape: {positions.shape}')
print(f'velocities shape: {velocities.shape}')
print(f'First satellite  NORAD {norad_ids[0]}:')
print(f'  position  = {positions[0].round(2)} km')
print(f'  velocity  = {velocities[0].round(4)} km/s')

### 4b. Per-satellite dict (same shape as `propagate_all`)

In [ ]:
pos_dict = snap.positions   # {norad_id: (pos_km, vel_km_s)}

first_id = norad_ids[0]
pos, vel = pos_dict[first_id]
print(f'Satellite {first_id}:  pos={tuple(round(x,2) for x in pos)}  vel={tuple(round(x,4) for x in vel)}')

### 4c. Filter by operator

In [ ]:
if snap.operator_names:
    op = snap.operator_names[0]
    sats = snap.get_operator_satellites(op)
    print(f'Operator "{op}": {len(sats)} satellites')
    for s in sats[:3]:
        print(f'  {s.label:25s}  alt={s.altitude_km:7.1f} km  speed={s.speed_km_s:.3f} km/s')

### 4d. Active conjunctions

In [ ]:
active = snap.active_conjunctions
print(f'Active conjunctions (Pc ≥ 1e-4): {len(active)}')
for c in active:
    print(f'  {c.pair_label}')
    print(f'    miss dist = {c.miss_distance_km:.4f} km')
    print(f'    Pc        = {c.pc:.3e}')
    print(f'    geometry  = {c.geometry_class}')
    print(f'    TCA       = {c.tca_offset_s:.0f} s from window centre')

### 4e. Conjunctions as a pandas DataFrame (requires pandas)

In [ ]:
df = env.conjunctions_dataframe()
df

## 5. Seek to a specific time offset

`seek(offset_s)` jumps the simulation clock to exactly `offset_s` seconds
after the epoch, without stepping. Useful for jumping to a known encounter time.

In [ ]:
env.seek(30 * 60)   # jump to T+30 minutes
snap30 = env.snapshot()
print(f'Time after seek: {snap30.sim_time}')
print(f'Active conjunctions: {len(snap30.active_conjunctions)}')

## 6. Reset and replay

In [ ]:
env.reset()   # back to T=0
print(f'After reset: step_index={env._step_index}, sim_time={env.sim_time}')

## 7. Time-loop example — watch conjunctions evolve over an hour

In [ ]:
import pandas as pd

env.reset()
history = []

# Step through 12 ticks of 5 minutes each = 1 hour
for _ in range(12):
    snap = env.step()
    for c in snap.conjunctions:
        history.append({
            'sim_time'       : snap.sim_time,
            'step'           : snap.step_index,
            'pair'           : c.pair_label,
            'miss_dist_km'   : c.miss_distance_km,
            'pc'             : c.pc,
            'above_threshold': c.pc >= 1e-4,
        })

history_df = pd.DataFrame(history)
print(f'Total conjunction records collected: {len(history_df)}')
history_df.head(10)

## 8. Export to CSV for external use

In [ ]:
# export_path = 'conjunction_history.csv'
# history_df.to_csv(export_path, index=False)
# print(f'Written to {export_path}')